In [1]:
%pip install python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

KEY: str = os.environ["KOREAN_DICT_KEY"]

print(KEY[:4] + "***")

29A9***


Q1 - 작성한 파이썬 코드

In [14]:
import requests
import json
import os
from dotenv import load_dotenv

# (a) search_word 함수 작성
def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    url = "https://opendict.korean.go.kr/api/search"
    params = {
        "key": KEY,
        "q": q,
        "req_type": "json",
        "num": num,
        "start": start,
        "type1": "word"
    }
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

# (b) 응답 구조 살펴보기
data = search_word("김치")
print("--- JSON 응답 구조 ---")
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])
print("\n" + "="*50 + "\n")

# (c) 원하는 정보만 추출
channel = data.get("channel", {})
total = channel.get("total", 0)
items = channel.get("item", [])
n = len(items)

# (i) 전체 결과 수와 이 페이지 항목 수 출력
print(f"총 {total}건, 이 페이지 {n}건\n")

# (ii) 첫 5개 항목 추출 및 출력
for item in items[:5]:
    word = item.get("word", "")
    sense = item.get("sense", {})
    
    if isinstance(sense, list) and len(sense) > 0:
        # ⚠️ 뒤에 or "품사 없음"을 붙여서 빈 문자열("")일 때를 방어합니다!
        pos = sense[0].get("pos") or "품사 없음"
        definition = sense[0].get("definition", "")
    elif isinstance(sense, dict):
        pos = sense.get("pos") or "품사 없음"
        definition = sense.get("definition", "")
    else:
        pos = "품사 없음"
        definition = ""
        
    print(f"{word} ({pos}) -> {definition[:40]}")

--- JSON 응답 구조 ---
{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


총 328건, 이 페이지 10건

김치 (명사) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


Q1 - 실행결과

--- JSON 응답 구조 ---
{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          

==================================================

총 328건, 이 페이지 10건

김치 (명사) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (명사) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (명사) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) -> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 

Q1 - 설명

json.dumps에서 ensure_ascii=False를 빼면 출력이 어떻게 달라지는가?
: ensure_ascii=False 옵션을 제거하면 한글 텍스트가 유니코드 이스케이프 시퀀스(예: \uae40\uce58) 형태로 강제 변환되어 출력되므로 사람이 내용을 정상적으로 읽고 파악할 수 없게 된다.

Q2 - 작성한 파이썬 코드

In [13]:
import time
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥", 
    "국수", "떡볶이", "불고기", "비빔밥"
]

all_items: list[dict] = []

print("--- (a) 검색어별 결과 수 ---")
for word in words:
    data = search_word(word)
    
    total: int = data.get("channel", {}).get("total", 0)
    print(f"{word}: {total}건")
    
    items: list[dict] = data.get("channel", {}).get("item", [])
    all_items.extend(items)
    
    time.sleep(0.3)

print("\n--- (b) 품사 빈도 상위 3개 ---")
pos_list: list[str] = []

for item in all_items:
    sense = item.get("sense", {})
    
    if isinstance(sense, list) and len(sense) > 0:
        pos = sense[0].get("pos") or "(미상)"
    elif isinstance(sense, dict):
        pos = sense.get("pos") or "(미상)"
    else:
        pos = "(미상)"
        
    pos_list.append(pos)

# 빈도 계산 및 상위 3개 추출
counter = Counter(pos_list)
for pos, count in counter.most_common(3):
    print(f"{pos}: {count}회")

--- (a) 검색어별 결과 수 ---
김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건

--- (b) 품사 빈도 상위 3개 ---
명사: 60회
(미상): 19회
어미: 1회


Q2 - 실행결과

--- (a) 검색어별 결과 수 ---
김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건

--- (b) 품사 빈도 상위 3개 ---
명사: 60회
(미상): 19회
어미: 1회

Q3 - 설명

가장 흔한 품사는 명사이다. 검색어로 사용된 '김치', '라면', '만두' 등의 8개 단어가 모두 일상생활에서 구체적인 사물의 이름을 지칭하는 체언이기 때문에 검색 결과 항목들 역시 명사가 많은 비율을 차지하게 된다.